In [1]:
class Variable():
    def __init__(self, data):
        self.data = data

In [2]:
import numpy as np
data = np.array([1.0])
x = Variable(data)
print(x.data)

[1.]


In [3]:
x.data = np.array([[[1.0]*3]*2]*3)
print(x.data)

[[[1. 1. 1.]
  [1. 1. 1.]]

 [[1. 1. 1.]
  [1. 1. 1.]]

 [[1. 1. 1.]
  [1. 1. 1.]]]


In [9]:
class Function:
    def __call__(self, input):
        x = input.data
        y = self.forward(x)
        output = Variable(y)
        return output
    def forward(self, x):
        raise NotImplementedError()

In [10]:
class Square(Function):
    def forward(self, x):
        return x ** 2

In [11]:
class Exp(Function):
    def forward(self, x):
        return np.exp(x)

In [45]:
x = Variable(np.array(1.0))
f = Exp()
y = f(x)
print(y.data)

2.718281828459045


In [20]:
A = Square()
B = Exp()
C = Square()

x = Variable(np.array(0.5))
a = A(x)
b = B(a)
y = C(b)
print(y.data)

1.648721270700128


In [15]:
def numerical_diff(f, x, eps=1e-4):
    #1.decoder
    x0 = Variable(x.data - eps)
    x1 = Variable(x.data + eps)
    y1, y0 = f(x1), f(x0)
    return (y1.data - y0.data) / (2 * eps)

In [21]:
f = Square()
x = Variable(np.array(2))
dy = numerical_diff(f, x, eps=1e-7)
print(dy)

3.9999999956741306


In [4]:
class Variable:
    def __init__(self, data):
        self.data = data

In [7]:
import numpy as np
x = Variable(np.array(1))
print(x.data)
print(type(x.data))

1
<class 'numpy.ndarray'>


In [22]:
def f(x):
    A = Square()
    B = Exp()
    C = Square()
    return C(B(A(x)))

x = Variable(np.array(0.5))
dy = numerical_diff(f, x)
print(dy)

3.2974426293330694


In [28]:
a = x.data
4 * a * (np.exp(a**2))**2-dy

-8.793281347507786e-08

In [32]:
class Varibale:
    def __init__(self, data):
        self.data = data
        self.grad = None

In [39]:
class Function:
    def __call__(self, input):
        x = input.data
        y = self.forward(x)
        output = Variable(y)
        self.input = input
        return output
    def forward(self, x):
        raise NotImplementedError
    def backward(self, gy):
        raise NotImplementedError
        
class Square(Function):
    def forward(self, x):
        return x ** 2
    def backward(self, gy):
        x = self.input.data
        gx = 2 * x * gy
        return gx
    
class Exp(Function):
    def forward(self, x):
        return np.exp(x)
    def backward(self, gy):
        x = self.input.data
        gx = np.exp(x) * gy
        return gx

In [47]:
A = Square()
x = Variable(np.array(0.5))
print(x.data)
y = A(x)
print(y.data)


0.5
0.25


In [48]:
A = Square()
B = Exp()
C = Square()
x = Variable(np.array(0.5))
a = A(x)
b = B(a)
y = C(b)
print(y.data)

y.grad = np.array(1.0)
b.grad = C.backward(y.grad)
a.grad = B.backward(b.grad)
x.grad = A.backward(a.grad)
print(x.grad)

1.648721270700128
3.297442541400256


In [35]:
class Variable:
    def __init__(self, data):
        self.data = data
        self.grad = None
        self.creator = None

[0.]


In [22]:
import numpy as np
class Variable:
    def __init__(self, data):
        self.data = data
        self.grad = None
        
class Function:
    def __call__(self ,input):
        x = input.data
        y = self.forward(x)
        output = Variable(y)
        self.input = input#remember paramters to cal gradient
        return output
    def forward(self, x):
        raise NotImplementedError
    def backward(self, gy):
        raise NotImplementedError
class Square(Function):
    def forward(self, x):
        return x**2
    def backward(self, gy):
        x = self.input.data
        gx = gy * 2 * x
        return gx
class Exp(Function):
    def forward(self ,x):
        return np.exp(x)
    def backward(self, gy):
        x = self.input.data
        gx = gy * np.exp(x)
        return gx

In [31]:
A = Square() 
B = Exp()
C = Square()

x = Variable(np.array(0.5))
a = A(x)
b = B(a)
y = C(b)

y.grad = np.array(1.0)
b.grad = C.backward(y.grad)
a.grad = B.backward(b.grad)
x.grad = A.backward(a.grad)
print(x.grad)

3.297442541400256


In [45]:
#函数列表
funs = [Square(),Exp(),Square()]
#x 的初始值
x = Variable(np.array(0.5))
#以下代码全自动不用改
var = list()
var.append(x)
for i in range(len(funs)):
    f = funs[i]
    x = var[i]
    y = f(x)
    var.append(y)
    
var[-1].grad = np.array(1)
for i in range(len(funs) - 1, -1, -1):
    f = funs[i]
    var[i].grad = f.backward(var[i+1].grad)
print(var[0].grad)

    

3.297442541400256


In [20]:
import numpy as np
class Variable:
    def __init__(self, data):
        self.data = data
        self.grad = None
        self.creator = None
    def set_creator(self, func):
        self.creator = func
    def backward(self):
        funcs = [self.creator]
        while funcs:
            f = funcs.pop()
            x, y = f.input, f.output
            x.grad = f.backward(y.grad)
            if x.creator is not None:
                funcs.append(x.creator)
        
        
class Function:
    def __call__(self ,input):
        x = input.data
        y = self.forward(x)
        output = Variable(y)
        output.set_creator(self)#remember creator function
        self.input = input#remember paramters input
        self.output = output#remember paramters output
        return output
    def forward(self, x):
        raise NotImplementedError
    def backward(self, gy):
        raise NotImplementedError
class Square(Function):
    def forward(self, x):
        return x**2
    def backward(self, gy):
        x = self.input.data
        gx = gy * 2 * x
        return gx
class Exp(Function):
    def forward(self ,x):
        return np.exp(x)
    def backward(self, gy):
        x = self.input.data
        gx = gy * np.exp(x)
        return gx
def square(x):
    f = Square()
    return f(x)
def exp(x):
    f = Exp()
    return f(x)

In [22]:
A = Square() 
B = Exp()
C = Square()

x = Variable(np.array(0.5))
a = A(x)
b = B(a)
y = C(b)

y.grad = np.array(1)
y.backward()
print(x.grad)

3.297442541400256


None


None


In [65]:
y.grad = np.array(1)

C = y.creator
b = C.input
b.grad = C.backward(y.grad)

B = b.creator
a = B.input
a.grad = B.backward(b.grad)

A = a.creator
x = A.input
x.grad = A.backward(a.grad)
print(x.grad)

3.297442541400256


In [18]:
import numpy as np
class Variable:
    def __init__(self, data):
        if data is not None:
            if not isinstance(data, np.ndarray):
                raise TypeError('{} is not supported'.format(type(data)))
                
        self.data = data
        self.grad = None
        self.creator = None
    def set_creator(self, func):
        self.creator = func
    def backward(self):
        #对当前的y_grad进行初始化
        if self.grad is None:
            self.grad = np.ones_like(self.data)
            
        funcs = [self.creator]
        while funcs:
            f = funcs.pop()
            x, y = f.input, f.output
            x.grad = f.backward(y.grad)
            if x.creator is not None:
                funcs.append(x.creator)
        
        
class Function:
    def __call__(self ,input):
        x = input.data
        y = self.forward(x)
        output = Variable(y)
        output.set_creator(self)#remember creator function
        self.input = input#remember paramters input
        self.output = output#remember paramters output
        return output
    def forward(self, x):
        raise NotImplementedError
    def backward(self, gy):
        raise NotImplementedError
class Square(Function):
    def forward(self, x):
        return x**2
    def backward(self, gy):
        x = self.input.data
        gx = gy * 2 * x
        return gx
class Exp(Function):
    def forward(self ,x):
        return np.exp(x)
    def backward(self, gy):
        x = self.input.data
        gx = gy * np.exp(x)
        return gx
def square(x):
    f = Square()
    return f(x)
def exp(x):
    f = Exp()
    return f(x)

In [107]:
Variable(None)


In [23]:
x = Variable(np.array(0.5))
a = square(x)
b = exp(a)
y = square(b)

#y.grad = np.array(1)
y.backward()
print(x.grad)

TypeError: unsupported operand type(s) for *: 'NoneType' and 'int'

In [13]:
import numpy as np

class Variable:
    def __init__(self, data):
        self.data = data
        self.grad = None
        self.creator = None
    def set_creator(self, func):
        self.creator = func
class Function:
    def __call__(self, input):
        x = input.data

        y = self.forward(x)

        output = Variable(y)
        #remember
        self.input = input
        self.output = output
        output.set_creator(self)
    def forward(self, x):
        raise NotImplementedError
        # return 当前模块的原函数
    def backward(self, gy):
        raise NotImplementedError
        # x = self.input.data
        # gx = 当前模块的导数(x) * gy
        # return gx
class Square(Function):
    def forward(self, x):
        return x**2
    def backward(self, gy):
        x = self.input.data
        gx = 2 * x * gy
        return gx
class Exp(Function):
    def forward(self, x):
        return np.exp(x)
    def backward(self, gy):
        x = self.input.data
        gx = np.exp(x) * gy
        return gx

In [25]:
#test
A = Square()
B = Exp()
C = Square()

x = Variable(np.array(0.5))
print(type(x))
a = A(x)
print(type(a))
b = B(a)
y = C(y)
print(y.data)

<class '__main__.Variable'>
<class '__main__.Variable'>
2.7182818284590446


In [24]:
import numpy as np
class Variable:
    def __init__(self, data):
        self.data = data
        self.grad = None
        self.creator = None
    def set_creator(self, func):
        self.creator = func
    def backward(self):
        funcs = [self.creator]
        while funcs:
            f = funcs.pop()
            x, y = f.input, f.output
            x.grad = f.backward(y.grad)
            if x.creator is not None:
                funcs.append(x.creator)
        
        
class Function:
    def __call__(self ,input):
        x = input.data
        y = self.forward(x)
        output = Variable(y)
        output.set_creator(self)#remember creator function
        self.input = input#remember paramters input
        self.output = output#remember paramters output
        return output
    def forward(self, x):
        raise NotImplementedError
    def backward(self, gy):
        raise NotImplementedError
class Square(Function):
    def forward(self, x):
        return x**2
    def backward(self, gy):
        x = self.input.data
        gx = gy * 2 * x
        return gx
class Exp(Function):
    def forward(self ,x):
        return np.exp(x)
    def backward(self, gy):
        x = self.input.data
        gx = gy * np.exp(x)
        return gx
def square(x):
    f = Square()
    return f(x)
def exp(x):
    f = Exp()
    return f(x)